# 콘텐츠 기반 필터링 따라가기

이 노트북은 **이 유저의 학습 평점 + 영화 속성 + 유저 속성**만 써서 별점을 예측합니다.
다른 유저가 그 영화에 준 평점은 쓰지 않습니다.

예전 방식은 장르 코사인을 `3 + 2 × 유사도`로 별점처럼 펼쳤습니다. 지금은 기준을 **그 사람 평균**에 두고, 나머지를 **Ridge**가 학습합니다.

$$\text{예측} = \text{유저 평균} + \text{Ridge}(\text{장르, 연도, 취향 매칭, 성별}\times\text{장르, 나이}\times\text{연도, }\ldots)$$

- 영화: `movies.dat` (장르, 제목 안 개봉연도)
- 평점: `ratings.dat`의 **train만** (취향 벡터·Ridge 학습). timestamp가 있으면 최근 평점에 더 큰 가중
- 유저: `users.dat` (성별, 나이, 직업). 전역 모델이라서 영화와의 곱으로만 넣습니다
- 쓰지 않는 것: 다른 사람 평점, 영화 평균 별점, zip code


## 0. 라이브러리와 경로


In [12]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

ROOT = Path.cwd()
if (ROOT / "src").exists() is False:
    ROOT = ROOT.parent

RAW_DIR = ROOT / "preprocessing_and_data" / "movielens-1m"

print("프로젝트 루트:", ROOT)
print("movies.dat:", (RAW_DIR / "movies.dat").exists())
print("ratings.dat:", (RAW_DIR / "ratings.dat").exists())
print("users.dat:", (RAW_DIR / "users.dat").exists())


프로젝트 루트: /Users/chaehee/Documents/school/ML/machine_learning_term_project
movies.dat: True
ratings.dat: True
users.dat: True


## 1. 영화 피처: 장르 TF-IDF + 개봉연도

장르는 `|`로 나눈 뒤 0/1 표를 만들고, 드문 장르에 가중을 주기 위해 TF-IDF처럼 IDF를 곱한 다음 길이 1로 맞춥니다.
제목 `Toy Story (1995)`에서 연도를 뽑아 표준화합니다.


In [13]:
YEAR_RE = re.compile(r"\((\d{4})\)")


def parse_year(title):
    match = YEAR_RE.search(str(title))
    return float(match.group(1)) if match else np.nan


movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["item_id", "title", "genres"],
)
genre_lists = movies["genres"].fillna("").str.split("|")
genre_cols = sorted({g for row in genre_lists for g in row if g})
movie_multihot = np.zeros((len(movies), len(genre_cols)), dtype=float)
genre_index = {g: j for j, g in enumerate(genre_cols)}
for i, row in enumerate(genre_lists):
    for genre in row:
        if genre in genre_index:
            movie_multihot[i, genre_index[genre]] = 1.0

n_movies = len(movies)
idf = np.log((n_movies + 1.0) / (movie_multihot.sum(axis=0) + 1.0)) + 1.0
tfidf = movie_multihot * idf
tfidf_norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
tfidf_norms[tfidf_norms == 0] = 1.0
item_tfidf = tfidf / tfidf_norms

years = movies["title"].map(parse_year).to_numpy(dtype=float)
year_mean = np.nanmean(years)
years = np.where(np.isfinite(years), years, year_mean)
year_norm = (years - year_mean) / np.nanstd(years)

item_ids = movies["item_id"].to_numpy()
titles = movies["title"].to_numpy()
item_index = {int(i): k for k, i in enumerate(item_ids)}

print("영화 수, 장르 수:", item_tfidf.shape)
print("장르:", genre_cols)
print("연도 범위:", int(years.min()), "~", int(years.max()))
movies.head()


영화 수, 장르 수: (3883, 18)
장르: ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
연도 범위: 1919 ~ 2000


,item_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


## 2. 평점·유저 읽기

평점은 누가 몇 점을 줬는지와, 언제 줬는지(최근성 가중)를 씁니다.
`users.dat`의 성별·나이·직업은 나중에 영화 피처와 곱해서 넣습니다. 유저별 모델에서는 이 값들이 상수라 쓸모가 없고, 전 유저를 한 Ridge에 넣을 때만 의미가 있습니다.


In [14]:
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["user_id", "item_id", "rating", "timestamp"],
)
users = pd.read_csv(
    RAW_DIR / "users.dat",
    sep="::",
    engine="python",
    encoding="ISO-8859-1",
    names=["user_id", "gender", "age", "occupation", "zip"],
)
print(ratings.shape, users.shape)
print(ratings["rating"].value_counts().sort_index())
users.head()


(1000209, 4) (6040, 5)
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


,user_id,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


## 3. 학습/테스트 나누기

사용자마다 일부 평점을 테스트로 떼고, 취향과 Ridge는 **train만**으로 만듭니다.


In [15]:
def split_by_user(ratings, test_size=0.2, random_state=42):
    frame = ratings.reset_index(drop=True)
    rng = np.random.default_rng(random_state)
    mask = np.zeros(len(frame), dtype=bool)
    for positions in frame.groupby("user_id", sort=False).indices.values():
        if len(positions) > 1:
            count = min(len(positions) - 1, max(1, int(len(positions) * test_size)))
            mask[rng.choice(positions, count, replace=False)] = True
    return frame.loc[~mask].copy(), frame.loc[mask].copy()


train, test = split_by_user(ratings)
print("train:", len(train), "test:", len(test))
print("학습 사용자 수:", train["user_id"].nunique())


train: 802553 test: 197656
학습 사용자 수: 6040


## 4. 유저 평균과 취향 벡터 (학습 평점만)

1. 그 사람 평균을 기준으로 둡니다. `residual = rating - user_mean`
2. 최근 평점일수록 가중을 크게 줍니다.
3. 그 가중 잔차를 영화 장르 벡터에 곱해 더하면, "이 사람이 자기 평균보다 높게 준 장르"가 취향 벡터가 됩니다.


In [16]:
N_OCCUPATIONS = 21
HALF_LIFE_DAYS = 365.0

train = train[train["item_id"].isin(item_index)].copy()
user_means = train.groupby("user_id")["rating"].mean()
global_mean = float(train["rating"].mean())
rating_lo, rating_hi = float(train["rating"].min()), float(train["rating"].max())

age_days = (float(train["timestamp"].max()) - train["timestamp"].astype(float)) / 86400.0
train["recency_w"] = 0.5 ** (age_days / HALF_LIFE_DAYS)
train["user_mean"] = train["user_id"].map(user_means)
train["residual"] = train["rating"] - train["user_mean"]
train["weighted_residual"] = train["recency_w"] * train["residual"]

user_ids = np.array(sorted(user_means.index), dtype=int)
user_index = {int(u): i for i, u in enumerate(user_ids)}
user_mean_arr = np.array([user_means[int(u)] for u in user_ids], dtype=float)

profiles = np.zeros((len(user_ids), len(genre_cols)), dtype=float)
u_idx = train["user_id"].map(user_index).to_numpy()
i_idx = train["item_id"].map(item_index).to_numpy()
np.add.at(profiles, u_idx, item_tfidf[i_idx] * train["weighted_residual"].to_numpy()[:, None])
profile_norms = np.linalg.norm(profiles, axis=1, keepdims=True)
profile_norms[profile_norms == 0] = 1.0
user_profiles = profiles / profile_norms

print("유저 프로필 크기:", user_profiles.shape)
print("전체 평균 별점:", round(global_mean, 3))
print("유저 1 평균:", round(float(user_means.get(1, np.nan)), 3))
pd.DataFrame(user_profiles, index=user_ids, columns=genre_cols).head()


유저 프로필 크기: (6040, 18)
전체 평균 별점: 3.582
유저 1 평균: 4.209


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,0.017814,-0.076363,-0.426018,-0.037944,-0.002563,-0.049653,0.000000,0.732700,-0.079014,0.000000,0.000000,0.056652,0.000000,-0.421875,0.081001,-0.162366,0.226372,0.000000
2,-0.483286,0.087320,0.000000,0.000000,-0.188320,0.071789,0.000000,0.753692,0.000000,0.000000,-0.064564,0.000000,-0.040096,-0.003873,-0.114452,-0.343047,-0.014954,0.115516
3,0.122985,0.216964,0.131728,0.106936,-0.559566,0.000000,0.000000,0.073703,0.116507,0.000000,-0.187769,0.000000,-0.125946,-0.296179,-0.174482,0.047131,0.000000,0.636700
4,0.236041,-0.565716,0.000000,-0.044679,0.000000,0.143678,0.000000,0.157412,-0.060174,0.000000,0.114260,0.000000,0.000000,-0.039210,-0.592705,0.011509,-0.442351,0.107296
5,-0.438967,-0.015538,0.133120,0.099713,0.569334,0.196278,0.125371,-0.496221,0.000000,0.104638,-0.100222,0.061566,0.023966,-0.163248,0.007335,-0.296119,0.113896,0.045415


## 5. Ridge 입력 만들기

한 행 = (이 유저, 이 영화). 목표값은 `residual` 즉 **별점 − 그 사람 평균**입니다.

| 피처 | 의미 |
|---|---|
| 영화 장르 TF-IDF | 이 영화가 무엇인가 |
| 개봉연도 | 새 영화/고전 |
| 취향 × 장르 | 이 사람 취향과 이 영화가 얼마나 겹치는가 |
| 남성 × 장르, 나이 × 연도/장르, 직업 × 연도 | 인구통계와 영화 내용의 상호작용 |

성별만 단독으로 넣으면 그 유저의 모든 영화에 같은 값이 나와서, 이미 있는 유저 평균과 겹칩니다. 그래서 **영화와 곱한 값만** 씁니다.


In [17]:
users = users.set_index("user_id")
is_male = np.zeros(len(user_ids), dtype=float)
age_scaled = np.zeros(len(user_ids), dtype=float)
occupation = np.zeros((len(user_ids), N_OCCUPATIONS), dtype=float)
for i, user_id in enumerate(user_ids):
    if int(user_id) not in users.index:
        continue
    row = users.loc[int(user_id)]
    is_male[i] = 1.0 if str(row["gender"]).upper() == "M" else 0.0
    age_scaled[i] = (float(row["age"]) - 25.0) / 25.0
    occ = int(row["occupation"])
    if 0 <= occ < N_OCCUPATIONS:
        occupation[i, occ] = 1.0


def make_features(user_idx, item_idx):
    tfidf = item_tfidf[item_idx]
    year = year_norm[item_idx][:, None]
    profile = user_profiles[user_idx]
    male = is_male[user_idx][:, None]
    age = age_scaled[user_idx][:, None]
    occ = occupation[user_idx]
    return np.hstack([
        tfidf,
        year,
        profile * tfidf,
        male * tfidf,
        age * year,
        age * tfidf,
        occ * year,
    ]).astype(np.float32)


sample_n = min(30000, len(u_idx))
sample_idx = np.random.default_rng(42).choice(len(u_idx), size=sample_n, replace=False)
X_train = make_features(u_idx[sample_idx], i_idx[sample_idx])
y_train = train["residual"].to_numpy(dtype=np.float32)[sample_idx]
w_train = train["recency_w"].to_numpy(dtype=float)[sample_idx]
print("원리 확인용 샘플:", X_train.shape, "(전체 학습은 run_content_based.py)")


원리 확인용 샘플: (30000, 95) (전체 학습은 run_content_based.py)


## 6. Ridge 학습과 예측

`3 + 2 × cosine` 대신, 학습 데이터에서 오차가 작아지도록 가중치를 찾습니다.
예측은 항상 그 사람 평균에서 출발합니다.


In [18]:
ridge = Ridge(alpha=10.0, fit_intercept=True, solver="cholesky")
ridge.fit(X_train, y_train, sample_weight=w_train)

seen = train.groupby("user_id")["item_id"].apply(set).to_dict()
item_means = train.groupby("item_id")["rating"].mean().to_dict()


def predict(user_id, item_id):
    u = user_index.get(int(user_id), -1)
    i = item_index.get(int(item_id), -1)
    if u >= 0 and i >= 0:
        residual = float(ridge.predict(make_features(np.array([u]), np.array([i])))[0])
        score = float(np.clip(user_mean_arr[u] + residual, rating_lo, rating_hi))
        n = int(np.sum((user_profiles[u] != 0) & (movie_multihot[i] != 0)))
        return score, "content", n
    if user_id in user_means.index:
        return float(user_means.loc[user_id]), "user_mean", 0
    if item_id in item_means:
        return float(item_means[item_id]), "item_mean", 0
    return global_mean, "global_mean", 0


example_user = int(user_ids[0])
example_item = int(item_ids[0])
score, method, n = predict(example_user, example_item)
print(f"user={example_user}, item={example_item} ({titles[0]})")
print("유저 평균:", round(float(user_means.loc[example_user]), 3))
print("예측:", round(score, 3), method, "기여 장르", n)


user=1, item=1 (Toy Story (1995))
유저 평균: 4.209
예측: 3.934 content 기여 장르 3


## 7. 추천: 안 본 영화 중 점수 상위 K

학습에서 이미 평가한 영화는 빼고, 점수는 높은 순, 같으면 `item_id` 낮은 순입니다.


In [20]:
def recommend(user_id, top_n=10):
    u = user_index.get(int(user_id), -1)
    already = seen.get(user_id, set())
    if u < 0:
        fallback = float(user_means.loc[user_id]) if user_id in user_means.index else global_mean
        rec = pd.DataFrame({
            "item_id": item_ids,
            "predicted_rating": fallback,
            "method": "user_mean" if user_id in user_means.index else "global_mean",
            "n_contributors": 0,
        })
    else:
        item_idx = np.arange(len(item_ids))
        residual = ridge.predict(make_features(np.full(len(item_ids), u), item_idx))
        scores = np.clip(user_mean_arr[u] + residual, rating_lo, rating_hi)
        n_c = np.sum((user_profiles[u] != 0) & (movie_multihot != 0), axis=1)
        rec = pd.DataFrame({
            "item_id": item_ids,
            "predicted_rating": scores,
            "method": "content",
            "n_contributors": n_c,
        })
    rec = rec[~rec["item_id"].isin(already)]
    rec = rec.sort_values(["predicted_rating", "item_id"], ascending=[False, True])
    rec = rec.head(top_n).reset_index(drop=True)
    rec["title"] = rec["item_id"].map(lambda i: titles[item_index[int(i)]])
    return rec


recommend(1, top_n=10)


,item_id,predicted_rating,method,n_contributors,title
0,557,5.0,content,1,Mamma Roma (1962)
1,668,5.0,content,1,Pather Panchali (1955)
2,669,5.0,content,1,Aparajito (1956)
3,670,5.0,content,1,"World of Apu, The (Apur Sansar) (1959)"
4,749,5.0,content,1,"Man from Down Under, The (1943)"
5,770,5.0,content,1,Costa Brava (1946)
6,820,5.0,content,1,"Death in the Garden (Mort en ce jardin, La) (1..."
7,854,5.0,content,1,"Ballad of Narayama, The (Narayama Bushiko) (1958)"
8,857,5.0,content,1,Crows and Sparrows (1949)
9,897,5.0,content,2,For Whom the Bell Tolls (1943)


## 8. (선택) 작은 평가

테스트 앞 2000개만 MAE/RMSE를 봅니다. 전체 평가는 `scripts/run_content_based.py`가 합니다.


In [19]:
sample_test = test.head(2000)
su = sample_test["user_id"].map(user_index).fillna(-1).astype(int).to_numpy()
si = sample_test["item_id"].map(item_index).fillna(-1).astype(int).to_numpy()
ok = (su >= 0) & (si >= 0)
preds = np.full(len(sample_test), global_mean, dtype=float)
preds[ok] = np.clip(
    user_mean_arr[su[ok]] + ridge.predict(make_features(su[ok], si[ok])),
    rating_lo,
    rating_hi,
)
err = preds - sample_test["rating"].to_numpy()
print("n =", len(sample_test))
print("MAE =", float(np.abs(err).mean()))
print("RMSE =", float(np.sqrt(np.mean(err ** 2))))


n = 2000
MAE = 0.7631504185351036
RMSE = 0.9620769932082867


## 정리

| 단계 | 어디서 |
|---|---|
| 영화 장르·연도 | `movies.dat` |
| 평점 나누기 | `ratings.dat` → train / test |
| 유저 평균·취향 벡터 | **train만** |
| 성별·나이·직업 | `users.dat` (영화와 곱해서만 사용) |
| 예측 | 유저 평균 + Ridge 보정, 1~5점으로 clip |

같은 내용이 `src/content_based.py`의 `fit` / `predict` / `recommend`에 들어가 있습니다. 노트북은 원리를 보려고 3만 행만 학습하고, 전체 train 평가는 아래를 실행하면 됩니다.

```bash
python scripts/run_content_based.py
```
